# How KV-Caches Make LLM Inference Fast
*Introduction to MLOps · companion notebook to the slides (MIT-style deck)*

The code in each part is the same code shown on the slides, with the slide numbers in the headings.

| Part | Slides | In the lecture |
|---|---|---|
| 1 · Load GPT-2 | 7–8 | on the slides |
| 2 · One token by hand | 10–13, 44 (backup) | on the slides |
| 3 · Autoregressive generation | 14 | on the slides |
| 4 · The simple loop | 16–19 | on the slides |
| 5 · Attention, one block and the causal mask | 21–23, 43 (backup) | on the slides (extra: see the real mask) |
| **6 · The loop with a cache** | **29–32** | **live at slide 30** |
| 7 · The price: memory | 41 (backup / next session) | at home |

**Before the lecture:** run the whole notebook once (*Kernel → Restart Kernel and Run All Cells*). This downloads GPT-2 and gives Part 6 the results of Part 4 to compare with. Then uncomment the `HF_HUB_OFFLINE` line below and zoom the browser to about 150 %.

## Setup

In [ ]:
import os
# os.environ["HF_HUB_OFFLINE"] = "1"   # lecture day: use the cached model, no network needed
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Part 1 · Load GPT-2

---
### Loading GPT-2
<span style="color:#1F4E8C">**Slide 7–8**</span> · Why does `c_attn` have 2304 outputs?

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()   # inference only: no dropout
print(model)

<details><summary><b>Answer: why 2304?</b></summary>

2304 = 3 × 768. One matrix multiply (`c_attn`) makes the **query, key and value** for every token. K and V are what we will cache.
</details>

## Part 2 · One token by hand

---
### Tokenize the prompt
<span style="color:#1F4E8C">**Slide 10**</span>

In [ ]:
inputs = tokenizer("Machine learning can", return_tensors="pt")
inputs

In [ ]:
[tokenizer.decode(t) for t in inputs["input_ids"][0]]   # the space is part of the token

---
### One forward pass
<span style="color:#1F4E8C">**Slide 11**</span> · shape = batch × tokens × vocabulary

In [ ]:
with torch.no_grad():
    logits = model(**inputs).logits
print(logits.shape)

---
### Pick the next token
<span style="color:#1F4E8C">**Slide 12**</span> · greedy decoding

In [ ]:
last_logits = logits[0, -1, :]          # last position only
next_token_id = last_logits.argmax()
next_token_id, tokenizer.decode(next_token_id)

*Greedy* always takes the best token. The top 10 show what *sampling* with a temperature could pick instead:

In [ ]:
[tokenizer.decode(t) for t in torch.topk(last_logits, k=10).indices]

---
### From the last vector to “be”
<span style="color:#1F4E8C">**Slide 44 (backup)**</span> · LM head → softmax → one token

Softmax turns the 50,257 scores into probabilities. These are the real numbers behind the bars on the slide:

In [ ]:
probs = torch.softmax(last_logits, dim=-1)
top = torch.topk(probs, k=5)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode(i)!r:10} {p.item():.3f}")
print("sum of all probabilities:", round(probs.sum().item(), 3))

The LM head is one linear layer, 768 → 50,257. We apply it by hand to the last vector after block 12 and get the same logits:

In [ ]:
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)
    h_last = out.hidden_states[-1][0, -1]        # last position, after block 12 (ln_f already applied)
    my_logits = model.lm_head(h_last)
print(h_last.shape, model.lm_head)
torch.allclose(my_logits, out.logits[0, -1], atol=1e-4)

---
### Append and repeat
<span style="color:#1F4E8C">**Slide 13**</span> · the whole sequence goes in again

In [ ]:
input_ids = inputs["input_ids"]
input_ids = torch.cat([input_ids, next_token_id.reshape(1, 1)], dim=1)
print(input_ids, input_ids.shape)

## Part 3 · Autoregressive generation

---
### Autoregressive: one token at a time
<span style="color:#1F4E8C">**Slide 14**</span> · each new token goes back into the input

In [ ]:
ids = tokenizer("Machine learning can", return_tensors="pt")["input_ids"]
for step in range(4):
    with torch.no_grad():
        next_id = model(input_ids=ids).logits[0, -1].argmax()
    print(f"step {step + 1}: {tokenizer.decode(ids[0])!r:40} -> {tokenizer.decode(next_id)!r}")
    ids = torch.cat([ids, next_id.reshape(1, 1)], dim=1)   # feed the new token back

## Part 4 · The simple loop (no cache)

---
### The simple loop
<span style="color:#1F4E8C">**Slide 16–18**</span> · flat or rising?

In [ ]:
def generate_token(inputs):
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits[0, -1, :].argmax()

N_NEW_TOKENS = 50
_ = generate_token(inputs)   # warm-up, not timed

In [ ]:
generated_tokens = []
next_inputs = inputs          # start: the tokenized prompt "Machine learning can"
durations_s = []
for _ in range(N_NEW_TOKENS):
    t0 = time.perf_counter()
    next_token_id = generate_token(next_inputs)
    durations_s.append(time.perf_counter() - t0)
    ids, mask = next_inputs["input_ids"], next_inputs["attention_mask"]
    next_inputs = {                                        # the whole sequence, every step
        "input_ids": torch.cat([ids, next_token_id.reshape(1, 1)], dim=1),
        "attention_mask": torch.cat([mask, torch.tensor([[1]])], dim=1),
    }
    generated_tokens.append(tokenizer.decode(next_token_id))

generated_tokens_no_cache = list(generated_tokens)
print(f"{sum(durations_s):.3f} s for {N_NEW_TOKENS} tokens")
print("Machine learning can" + "".join(generated_tokens))

---
### Result: rising
<span style="color:#1F4E8C">**Slide 19**</span>

In [ ]:
plt.plot(np.array(durations_s) * 1000, marker="o")
plt.xlabel("Token number")
plt.ylabel("Time per token (ms)")
plt.title("Without KV-cache")
plt.show()

## Part 5 · Attention, one block and the causal mask

---
### After attention: from y₂ to “be”
<span style="color:#1F4E8C">**Slide 43 (backup)**</span> · one block = LayerNorm → attention → add → LayerNorm → MLP → add

Print one of the 12 blocks. You see the parts of the slide: `ln_1`, attention (`c_attn`, `c_proj`), `ln_2` and the MLP (`c_fc` 768 → 3072, `c_proj` 3072 → 768).

In [ ]:
print(len(model.transformer.h), "blocks")
model.transformer.h[0]

---
### The causal mask
<span style="color:#1F4E8C">**Slide 23**</span> · extra: the real attention weights of GPT-2

We ask GPT-2 for its attention weights on the 4 tokens *Machine learning can be*.
Row = the token that looks, column = the token it looks at. Everything above the diagonal is **0**: a token only looks back.
(We load a second copy of the model with the simple "eager" attention, because it can return the weights.)

In [ ]:
model_eager = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").eval()
x = tokenizer("Machine learning can be", return_tensors="pt")
with torch.no_grad():
    att = model_eager(**x, output_attentions=True).attentions   # one tensor per layer

w = att[0][0, 0]            # layer 1, head 1: [tokens, tokens]
labels = [tokenizer.decode(t) for t in x["input_ids"][0]]
print(np.round(w.numpy(), 2))

plt.imshow(w, cmap="Reds", vmin=0, vmax=1)
plt.xticks(range(len(labels)), labels); plt.yticks(range(len(labels)), labels)
plt.xlabel("looks at (keys)"); plt.ylabel("token that looks")
plt.title("Causal mask: zeros above the diagonal")
plt.colorbar(); plt.show()

## Part 6 · The loop with a cache

<div style="background:#9B1B1B; color:white; padding:14px 20px; border-radius:6px">
<b>▶ LIVE DEMO · slide 30</b> — switch here from the slides, run the cells of this part from top to bottom, then go back to slide 31.
</div>

---
### The loop with a cache
<span style="color:#1F4E8C">**Slide 29**</span> · only the new token · full attention mask · the cache

In [ ]:
def generate_token_with_past(inputs):
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits[0, -1, :].argmax(), outputs.past_key_values

In [ ]:
generated_tokens = []
next_inputs = inputs
durations_cached_s = []
for _ in range(N_NEW_TOKENS):
    t0 = time.perf_counter()
    next_token_id, past_key_values = generate_token_with_past(next_inputs)
    durations_cached_s.append(time.perf_counter() - t0)
    next_inputs = {
        "input_ids": next_token_id.reshape((1, 1)),                                        # only the new token
        "attention_mask": torch.cat([next_inputs["attention_mask"], torch.tensor([[1]])], dim=1),  # still the full length
        "past_key_values": past_key_values,                                                # the cache
    }
    generated_tokens.append(tokenizer.decode(next_token_id))

print(f"{sum(durations_cached_s):.3f} s for {N_NEW_TOKENS} tokens")
print("Same text as without the cache:", generated_tokens == generated_tokens_no_cache)

---
### Result: flat
<span style="color:#1F4E8C">**Slide 31**</span>

In [ ]:
plt.plot(np.array(durations_s) * 1000, marker="o", label="without KV-cache")
plt.plot(np.array(durations_cached_s) * 1000, marker="o", label="with KV-cache")
plt.xlabel("Token number")
plt.ylabel("Time per token (ms)")
plt.title("Time per token")
plt.legend()
plt.savefig("kv_cache_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

---
### Result in numbers
<span style="color:#1F4E8C">**Slide 32**</span> · TTFT = step 1 (prefill) · TPOT = mean of the later steps (decode)

In [ ]:
def ttft_tpot(d):
    return d[0] * 1000, np.mean(d[1:]) * 1000

ttft_nc, tpot_nc = ttft_tpot(durations_s)
ttft_c, tpot_c = ttft_tpot(durations_cached_s)
print(f"{'':12s}{'without cache':>15s}{'with cache':>12s}")
print(f"{'TTFT (ms)':12s}{ttft_nc:15.1f}{ttft_c:12.1f}")
print(f"{'TPOT (ms)':12s}{tpot_nc:15.1f}{tpot_c:12.1f}")
print(f"{'total (s)':12s}{sum(durations_s):15.2f}{sum(durations_cached_s):12.2f}")
print(f"Time saved: {100 * (1 - sum(durations_cached_s) / sum(durations_s)):.0f} %")

*Optional:* the library does the same for you: `model.generate()` uses the KV cache by default.

In [ ]:
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=N_NEW_TOKENS, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0]))

## Part 7 · The price: memory

---
### The price: memory
<span style="color:#1F4E8C">**Slide 41 (backup)**</span> · 2 × layers × hidden × bytes per token

In [ ]:
def cache_tensors(pkv):
    # works for old (tuple) and new (Cache object) transformers versions
    if hasattr(pkv, "layers"):
        return [(l.keys, l.values) for l in pkv.layers]
    if hasattr(pkv, "key_cache"):
        return list(zip(pkv.key_cache, pkv.value_cache))
    return [(k, v) for k, v in pkv]

kv = cache_tensors(past_key_values)
n_bytes = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kv)
n_tokens = kv[0][0].shape[-2]
print(f"layers: {len(kv)}   K shape per layer: {tuple(kv[0][0].shape)}")
print(f"cached tokens: {n_tokens}   total: {n_bytes / 1024:.0f} KiB   per token: {n_bytes / n_tokens / 1024:.0f} KiB")
print(f"formula: 2 x 12 x 768 x 4 bytes = {2 * 12 * 768 * 4 / 1024:.0f} KiB per token")

---
## Homework (slide 36)
Hand in one page with your plots and a short answer to each task.

1. **Prompt length.** Use prompts with about 10, 100 and 500 tokens. Measure TTFT and TPOT, with and without the cache (Parts 4 and 6). Plot them. What grows, and why?
2. **Memory.** Read the size of `past_key_values` after 50 tokens (Part 7). Check it with 2 × layers × hidden × bytes × tokens.
3. **Decoding.** Use `model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7, top_k=50)`. Compare with greedy. Does `use_cache=False` change the text?
4. **Bonus.** Keep only the last 32 tokens in the cache (a sliding window). What happens to speed and to the text?

### More ideas
- Set `N_NEW_TOKENS = 200` and re-run Parts 4 and 6. How do the two curves change?
- With a GPU: `model.to("cuda")` and move `inputs` to `"cuda"`. Compare again.
- In Part 5, look at other layers and heads: `att[layer][0, head]`. The upper triangle is always 0.

## Further reading (slide 37)
- Vaswani et al., *Attention Is All You Need*, NeurIPS 2017.
- Radford et al., *Language Models are Unsupervised Multitask Learners* (GPT-2), OpenAI 2019.
- J. Alammar, *The Illustrated GPT-2* — https://jalammar.github.io/illustrated-gpt2/
- S. Raschka, *Understanding and Coding the KV Cache in LLMs from Scratch*, 2025 — https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms
- Hugging Face, *Cache strategies* — https://huggingface.co/docs/transformers/kv_cache
- DeepLearning.AI, *Efficiently Serving LLMs* (short course).
- Pope et al., *Efficiently Scaling Transformer Inference*, MLSys 2023.
- Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention* (vLLM), SOSP 2023.
- Shazeer, *Fast Transformer Decoding: One Write-Head is All You Need* (MQA), 2019; Ainslie et al., *GQA*, EMNLP 2023.